In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.options.display.max_rows = 300
import datetime as dt

# feature (vital, lab)

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype_v2/00.data_mimic//feature_data/"
infile = data_dir + "sofa.csv"
df_sofa = pd.read_csv(infile)
df_sofa["starttime"] = pd.to_datetime(df_sofa["starttime"], format='%d/%m/%Y %H:%M:%S')
df_sofa.loc[:, 'charttime'] = df_sofa.starttime + dt.timedelta(hours=0.5)
df_sofa= df_sofa.loc[:,["stay_id","charttime", "sofa_24hours", 'respiration_24hours', 'coagulation_24hours', 'liver_24hours','cardiovascular_24hours', 'cns_24hours', 'renal_24hours']]
df_sofa.columns = ["stay_id","charttime", 'sofa', 'respiration_sofa', 'coagulation_sofa', 'liver_sofa','cardiovascular_sofa', 'cns_sofa', 'renal_sofa']

data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype_v2/00.data_mimic//feature_data/"
infile = data_dir + "icu_lab_feature_merge.csv"
df_lab = pd.read_csv(infile)
df_lab = df_lab.drop(['subject_id', 'hadm_id',"troponin_t"],axis=1)
df_lab["charttime"] = pd.to_datetime(df_lab["charttime"], format='%d/%m/%Y %H:%M:%S')
df_lab = df_lab.drop(["glucose","temperature"],axis=1)

data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype_v2/00.data_mimic//feature_data/"
infile = data_dir + "icu_vitalsign_feature_merge.csv"
df_viatl = pd.read_csv(infile)
df_viatl = df_viatl.drop(['subject_id', 'hadm_id'],axis=1)
df_viatl["charttime"] = pd.to_datetime(df_viatl["charttime"], format='%d/%m/%Y %H:%M:%S')

df_fea = pd.merge(df_viatl, df_lab, how="outer",on=["stay_id","charttime"])

df_fea = df_fea.drop(['nrbc',"rdwsd"],axis=1)
df_fea.loc[df_fea["glucose"]>=1000,"glucose"] = np.nan
df_fea.loc[df_fea["pao2fio2ratio"]>=1000,"pao2fio2ratio"] = np.nan
df_fea.loc[df_fea["fibrinogen"]>=1000,"fibrinogen"] = np.nan
df_fea.loc[df_fea["platelet"]>=1000,"platelet"] = np.nan
df_fea.loc[df_fea["alt"]>=2000,"alt"] = np.nan
df_fea.loc[df_fea["alp"]>=2000,"alp"] = np.nan
df_fea.loc[df_fea["ast"]>=2000,"ast"] = np.nan
df_fea.loc[df_fea["ck_cpk"]>=2000,"ck_cpk"] = np.nan
df_fea.loc[df_fea["ggt"]>=1000,"ggt"] = np.nan
df_fea.loc[df_fea["ld_ldh"]>=2000,"ld_ldh"] = np.nan
df_fea.loc[df_fea["urineoutput"]<10,"urineoutput"] = np.nan
df_fea.loc[df_fea["urineoutput"]>1000,"urineoutput"] = np.nan
df_fea = df_fea.drop_duplicates()


In [ ]:
df_demo =  pd.read_csv("/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/feature_data/sk_icu_basicinfo.csv")
df_demo = df_demo.loc[:,["stay_id","intime","outtime"]]
df_demo["intime"] = pd.to_datetime(df_demo["intime"], format='%d/%m/%Y %H:%M:%S')
df_demo["outtime"] = pd.to_datetime(df_demo["outtime"], format='%d/%m/%Y %H:%M:%S')
df_demo = df_demo.drop_duplicates()

df_sofa = pd.merge(df_sofa,df_demo, how="inner", on="stay_id")
df_sofa = df_sofa[(df_sofa["charttime"]<=df_sofa["outtime"]) & (df_sofa["charttime"]>=df_sofa["intime"])]
df_sofa = df_sofa.drop(["intime","outtime"],axis=1)

df_fea = pd.merge(df_fea,df_demo, how="inner", on="stay_id")
df_fea = df_fea[(df_fea["charttime"]<=df_fea["outtime"]) & (df_fea["charttime"]>=df_fea["intime"])]
df_fea = df_fea.drop(["intime","outtime"],axis=1)

df_merge = pd.merge(df_fea, df_sofa, how="outer",on=["stay_id","charttime"])
df_merge = df_merge.sort_values(by=["stay_id","charttime"])
df_merge.to_csv("/public/hanl/jupyter_dir/kidney_sepsis_penotype_v2/00.data_mimic//feature_data/sk_icu_feature.csv", index=False)
#df_merge.to_csv("/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/feature_data/icu_feature_nofilt_toHB.csv", index=False)

In [ ]:
stop!

#  demo

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/disease_definition/"
infile = data_dir + "sk_sepsis_onset.csv"
df_sepsis = pd.read_csv(infile)

df_sepsis["sofa_time"] = pd.to_datetime(df_sepsis["sofa_time"], format='%d/%m/%Y %H:%M:%S')
df_sepsis["suspected_infection_time"] = pd.to_datetime(df_sepsis["suspected_infection_time"], format='%d/%m/%Y %H:%M:%S')
print(df_sepsis.shape[0])

df_sepsis= df_sepsis[ (df_sepsis["sofa_time"]>= df_sepsis["suspected_infection_time"]) ] #(df_sepsis["sofa_time"] >=df_sepsis["antibiotic_time"] )
print(df_sepsis.shape[0])
df_sepsis["sepsis_onset"] = df_sepsis["sofa_time"]

sepsis_lst = df_sepsis.stay_id.tolist()
print("sepsis患者数量为(by sepsis3)：",df_sepsis.shape[0])

df_sepsis  = df_sepsis.loc[:,["stay_id","sepsis3"]]
df_sepsis.columns = ["stay_id","issepsis3"]
df_sepsis["issepsis3"] = 1
df_sepsis.head(1)

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/disease_definition/"

infile = data_dir + "sk_aki_onset.csv"
df_aki = pd.read_csv(infile)
df_aki["charttime"] = pd.to_datetime(df_aki["charttime"], format='%d/%m/%Y %H:%M:%S')
df_aki = df_aki.rename(columns={'charttime':'AKI_onset'})

aki_lst = df_aki.stay_id.tolist()
print("aki患者数量为(by KDIGO)：",len(aki_lst))

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/disease_definition/"
infile = data_dir + "sk_ckd_diagnose.csv"
df_ckd = pd.read_csv(infile)
ckd_dict = {'N183   ':"ckd3" , 'N186   ':"ESRD", '5855   ':"ckd5", '5851   ':"ckd1", 'N185   ':"ckd5", 
            '5859   ':"ckd_unknow", '5856   ':"ESRD", '5854   ':"ckd4", '5852   ':"ckd2", 'N189   ':"ckd_unknow",
            'N184   ':"ckd4", 'N182   ':"ckd2", 'N181   ':"ckd1", '5853   ':"ckd3"}
df_ckd["CKD_stage"] = df_ckd["icd_code"].map(ckd_dict)
df_ckd = df_ckd.loc[:,[ 'stay_id', 'CKD_stage']].drop_duplicates()

tmp_df = pd.DataFrame()
df_ckd = df_ckd.sort_values(["stay_id","CKD_stage"])
for i,j in df_ckd.groupby("stay_id"):
    j = j.iloc[:1]
    tmp_df = pd.concat([tmp_df, j], axis=0)
df_ckd = tmp_df

ckd_lst = set(df_ckd.stay_id.tolist())
print("ckd患者数量为(by ICD)：",len(ckd_lst))
df_ckd.head(1)

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/feature_data/"
infile = data_dir + "apsiii.csv"
df_apsiii = pd.read_csv(infile)
df_apsiii = df_apsiii.loc[:,["stay_id","apsiii"]]

data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/treatment/"
infile = data_dir + "treatment_crrt.csv"
df_crrt = pd.read_csv(infile)
df_crrt["CRRT"] = 1
df_crrt = df_crrt.loc[:,["stay_id", "CRRT"]].drop_duplicates()
crrt_lst = df_crrt.stay_id.unique()
print("crrt:", len(crrt_lst))

data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/feature_data/"
infile = data_dir + "icu_basicinfo.csv"
df_demo = pd.read_csv(infile)
df_demo = df_demo.loc[:,['subject_id', 'hadm_id', 'stay_id', 'age', 'weight', 'gender', 'height',
       'race', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los',
       'dod', 'hospital_expire_flag']]

eth_map = {'MULTIPLE RACE/ETHNICITY':'MULTIPLE RACE', 'UNKNOWN':"UNKNOWN", 'WHITE':'WHITE',
       'BLACK/AFRICAN AMERICAN':'BLACK', 'UNABLE TO OBTAIN':"UNKNOWN",
       'ASIAN - ASIAN INDIAN':'ASIAN', 'WHITE - EASTERN EUROPEAN':'WHITE', 'OTHER':'OTHER',
       'WHITE - OTHER EUROPEAN':'WHITE', 'BLACK/CAPE VERDEAN':'BLACK',
       'HISPANIC/LATINO - PUERTO RICAN':'HISPANIC', 'ASIAN - CHINESE':'ASIAN',
       'BLACK/CARIBBEAN ISLAND':'BLACK', 'ASIAN':'ASIAN', 'HISPANIC/LATINO - DOMINICAN':'HISPANIC',
       'WHITE - BRAZILIAN':'WHITE', 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER':"HAWAIIAN",
       'WHITE - RUSSIAN':'WHITE', 'HISPANIC OR LATINO':'HISPANIC', 'PORTUGUESE':'PORTUGUESE',
       'AMERICAN INDIAN/ALASKA NATIVE':'AMERICAN_INDIAN', 'ASIAN - SOUTH EAST ASIAN':'ASIAN',
       'HISPANIC/LATINO - GUATEMALAN':'HISPANIC',
       'HISPANIC/LATINO - CENTRAL AMERICAN':'HISPANIC', 'SOUTH AMERICAN':'SOUTH_AMERICAN',
       'BLACK/AFRICAN':'BLACK', 'PATIENT DECLINED TO ANSWER':"UNKNOWN",
       'HISPANIC/LATINO - COLUMBIAN':'HISPANIC', 'ASIAN - KOREAN':'ASIAN',
       'HISPANIC/LATINO - MEXICAN':'HISPANIC', 'HISPANIC/LATINO - HONDURAN':'HISPANIC',
       'HISPANIC/LATINO - SALVADORAN':'HISPANIC'}

df_demo["race"] = df_demo["race"].map(eth_map)
df_demo = pd.merge(df_demo, df_crrt, how="left", on = "stay_id")
df_demo["CRRT"] = df_demo["CRRT"].fillna(0)
df_demo = pd.merge(df_demo, df_apsiii, how="left", on = "stay_id")

tmp_df = pd.DataFrame()
df_demo = df_demo.sort_values(["stay_id", "age"], ascending= False)
for i,j in df_demo.groupby("stay_id"):
    j = j.iloc[:1]
    tmp_df = pd.concat([tmp_df ,j], axis=0)
df_demo = tmp_df


unit_map = {'Cardiac Vascular Intensive Care Unit (CVICU)':"Cardiac",
           "Coronary Care Unit (CCU)":"Cardiac",
           "Medical Intensive Care Unit (MICU)":"Medical",
           "Medical/Surgical Intensive Care Unit (MICU/SICU)":"Medical/Surgical",
           'Neuro Intermediate':"Others",
           "Neuro Stepdown":"Others",
           "Neuro Surgical Intensive Care Unit (Neuro SICU)":"Surgical",
           "Surgical Intensive Care Unit (SICU)":"Surgical",
           "Trauma SICU (TSICU)":"Surgical"}
df_demo["first_careunit"] = df_demo["first_careunit"].map(unit_map)
df_demo["last_careunit"] = df_demo["last_careunit"].map(unit_map)


In [ ]:
# add_ckd info
df_demo["isckd"] = 0
df_demo.loc[df_demo["stay_id"].isin(ckd_lst), "isckd"] = 1

df_demo = pd.merge(df_demo, df_ckd, how="left", on = "stay_id")
df_demo["CKD_stage"] = df_demo["CKD_stage"].fillna("no_ckd")

# add_ckd info
df_demo["isaki"] = 0
df_demo.loc[df_demo["stay_id"].isin(aki_lst), "isaki"] = 1

df_demo = pd.merge(df_demo, df_aki, how="left", on = "stay_id")
df_demo["aki_stage"] = df_demo["aki_stage"].fillna(0)

# add sepsis info
df_demo = pd.merge(df_demo, df_sepsis, how="left", on = "stay_id")
df_demo["issepsis3"] = df_demo["issepsis3"].fillna(0)



In [ ]:
df_demo.to_csv("/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/feature_data/sk_icu_basicinfo.csv",index=False)

# RRT
- CRRT 三种模式
- IDH


## CRRT

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/treatment/"
infile = data_dir + "treatment_crrt.csv"
df_crrt = pd.read_csv(infile)
df_crrt.head(1)

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/treatment/"
infile = data_dir + "crrt_mode.csv"
df_crrt_mode = pd.read_csv(infile)
df_crrt_mode.head(1)

In [ ]:
df_crrt_add = pd.merge(df_crrt, df_crrt_mode, how="left", on="stay_id")
df_crrt_add["starttime"] = pd.to_datetime(df_crrt_add["starttime"], format='%d/%m/%Y %H:%M:%S')
df_crrt_add["endtime"] = pd.to_datetime(df_crrt_add["endtime"], format='%d/%m/%Y %H:%M:%S')
df_crrt_add["charttime"] = pd.to_datetime(df_crrt_add["charttime"], format='%d/%m/%Y %H:%M:%S')
df_crrt_add = df_crrt_add[(df_crrt_add["charttime"]<=df_crrt_add["endtime"]) & (df_crrt_add["charttime"]>=df_crrt_add["starttime"])]

tmp_df = pd.DataFrame(columns=['stay_id', 'num', 'starttime', 'endtime', 'crrt_mode_lst'])
for i,j in df_crrt_add.groupby(['stay_id', 'num', 'starttime', 'endtime']):
    tmp_df.loc[len(tmp_df)] = [j["stay_id"].tolist()[0], j["num"].tolist()[0], j["starttime"].tolist()[0], j["endtime"].tolist()[0], j.crrt_mode.unique()]
tmp_df['tag'] = range(len(tmp_df))

df_mode = tmp_df.set_index('tag').crrt_mode_lst.apply(pd.Series).stack().reset_index(level=0).rename(columns={0:'rrt_mode'})
df_crrt_add = pd.merge(tmp_df, df_mode, how="left", on="tag")
df_crrt_add = df_crrt_add.drop(["tag", "crrt_mode_lst"],axis=1)
df_crrt_add = df_crrt_add.sort_values(by=["stay_id","starttime", "endtime"])
# 每个ID 几乎只有一个CRRT mode，有两种的其中一种为nan； 唯一一位病人（ID 已删除）包含CVVHDF和CVVH两种模式

tmp_df = pd.DataFrame()
for i,j in df_crrt_add.groupby("stay_id"):
    j = j.fillna(method="bfill")
    j = j.fillna(method="ffill")
    tmp_df = pd.concat([tmp_df, j],axis=0)
df_crrt_add = tmp_df

df_crrt_add = df_crrt_add[df_crrt_add["rrt_mode"].isin(["CVVHDF", "CVVHD"])]

In [ ]:
print('使用CRRT病人数：', len(df_crrt_add.stay_id.unique()))
df_crrt_add["interval_H"] = (df_crrt_add["endtime"] - df_crrt_add["starttime"]).map(lambda x:x/np.timedelta64(1, 'h')).tolist()
df_crrt_add["interval_H"].hist(bins=20)

In [ ]:
df_crrt_add.head(1)

In [ ]:
'''tmp_lst = []
for i,j in df_crrt_add.groupby(["stay_id"]):
    if len(j.rrt_mode.unique())>1:
        tmp_lst.append(i)
        print(i, j.rrt_mode.unique())
#tmp_lst 包含两种CRRT模式'''

## IHD

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype/00.data/treatment/"
infile = data_dir + "rrt.csv"
df_rrt = pd.read_csv(infile)
IHD = df_rrt[df_rrt["dialysis_type"] == "IHD"]
IHD["charttime"] = pd.to_datetime(IHD["charttime"], format='%d/%m/%Y %H:%M:%S')
IHD["charttime_D"] = IHD["charttime"].dt.date


tmp_df = pd.DataFrame(columns=["stay_id", "starttime", "endtime","rrt_mode"])
for i,j in IHD.groupby(["stay_id", "charttime_D"]):
    tmp_df.loc[len(tmp_df)] = [j["stay_id"].tolist()[0], j["charttime"].tolist()[0], j["charttime"].tolist()[-1],"IHD"]
tmp_df["interval_H"] = (tmp_df["endtime"] - tmp_df["starttime"]).map(lambda x:x/np.timedelta64(1, 'h')).tolist()
tmp_df = tmp_df[tmp_df["interval_H"]> 0]

tmp_df = tmp_df.sort_values(by=["stay_id", "starttime"])
IHD = pd.DataFrame()

for i,j in tmp_df.groupby("stay_id"):
    j["num"] = range(len(j))
    IHD = pd.concat([IHD, j],axis=0)
IHD = IHD.drop(["interval_H"],axis=1)

In [ ]:
print('使用IHD病人数：', len(IHD.stay_id.unique()))
IHD["interval_H"] = (IHD["endtime"] - IHD["starttime"]).map(lambda x:x/np.timedelta64(1, 'h')).tolist()
IHD["interval_H"].hist(bins=24)

In [ ]:
IHD.head(1)

## IHD 和CRRT merge
- IHD
- df_crrt_add

In [ ]:
df_rrt = pd.concat([df_crrt_add, IHD],axis=0)
df_rrt = df_rrt.sort_values(["stay_id", "starttime"])
df_rrt = df_rrt.drop(["num"],axis=1)

df_bg_ed = pd.DataFrame(columns=["stay_id", "rrt_starttime", "rrt_endtime"])
for i,j in df_rrt.groupby("stay_id"):
    df_bg_ed.loc[len(df_bg_ed)] = [j["stay_id"].tolist()[0], j["starttime"].tolist()[0], j["endtime"].tolist()[-1]]

df_rrt = pd.merge(df_rrt, df_bg_ed, how="outer", on="stay_id")

In [ ]:
df_rrt.head(2)

In [ ]:
df_rrt.to_csv("sk_rrt.csv", index=False)

## stat

In [ ]:
len(list(set(IHD.stay_id.unique()).union(set(df_crrt_add.stay_id.unique()))))

In [ ]:
len(list(set(IHD.stay_id.unique()).intersection(set(df_crrt_add.stay_id.unique()))))